# 05 · Ensambles: homogéneo, heterogéneo e híbrido

**Tema 5 del proyecto.** Se combinan los 5 modelos entrenados en el notebook 04 (3 CNN y
2 transformers).

| Tipo | Definición | Ensamble construido |
|---|---|---|
| **Homogéneo** | Clasificadores del mismo tipo | Las 3 CNN (cnn_propia, EfficientNetB0, ResNet50V2) |
| **Heterogéneo** | Clasificadores de tipos distintos | Los 5 modelos: 3 CNN + 2 transformers |
| **Híbrido** | Combina ensambles homogéneos dentro de uno heterogéneo | Nivel 1: un ensamble de CNN y uno de transformers. Nivel 2: un meta-clasificador (*stacking*) que aprende a combinarlos |

**Reglas para no hacer trampa con test:**
- Los pesos de la votación ponderada y el meta-clasificador se ajustan **solo con validación**.
- Test se usa una vez, para reportar.

No se vuelve a entrenar ninguna red: se combinan las probabilidades guardadas. **No necesita
GPU** (Accelerator = None).

**Input:** la salida de `04-entrenamiento-final`, el dataset y `01-preparacion-datos` (este
último solo para mostrar las imágenes mal clasificadas).

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from scipy.stats import binomtest
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, f1_score, log_loss)
from sklearn.model_selection import StratifiedKFold, cross_val_predict

OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
SEED = 42
CNN = ["cnn_propia", "efficientnetb0", "resnet50v2"]
TRANSFORMERS = ["vit_b16", "deit_s"]


def find_dir_with(*names, base="/kaggle/input"):
    for root, _, files in os.walk(base):
        if all(n in files for n in names):
            return Path(root)
    raise FileNotFoundError(f"No se encontró {names}: revisa los Input del notebook")


FINAL = find_dir_with("test_y.npy", "val_y.npy")
SPLITS = find_dir_with("test.csv", "rice_data.py")
test_df = pd.read_csv(SPLITS / "test.csv")
CLASSES = test_df[["label_idx", "label"]].drop_duplicates().sort_values("label_idx")["label"].tolist()
y_val, y_test = np.load(FINAL / "val_y.npy"), np.load(FINAL / "test_y.npy")


def cargar(split, name):
    p = np.load(FINAL / f"{split}_probs_{name}.npy").astype("float64")
    return p / p.sum(axis=1, keepdims=True)


disponibles = [m for m in CNN + TRANSFORMERS
               if (FINAL / f"test_probs_{m}.npy").exists()
               and not np.isnan(np.load(FINAL / f"test_probs_{m}.npy")).any()]
P_val = {m: cargar("val", m) for m in disponibles}
P_test = {m: cargar("test", m) for m in disponibles}
CNN = [m for m in CNN if m in disponibles]
TRANSFORMERS = [m for m in TRANSFORMERS if m in disponibles]
print("Modelos disponibles:", disponibles)
print(f"val={len(y_val)}  test={len(y_test)}")

## 1. Punto de partida: cada modelo por separado (test)

In [ ]:
def ll(y, probs):
    # Log-loss evitando log(0): recorta y vuelve a normalizar las probabilidades.
    p = np.clip(probs, 1e-7, 1)
    return log_loss(y, p / p.sum(axis=1, keepdims=True), labels=list(range(len(CLASSES))))


PROBS = {}  # probabilidades en test de cada modelo y ensamble evaluado


def evaluar(nombre, tipo, probs, y=None):
    y = y_test if y is None else y
    PROBS[nombre] = probs
    pred = probs.argmax(1)
    return {"ensamble": nombre, "tipo": tipo, "accuracy": accuracy_score(y, pred),
            "f1_macro": f1_score(y, pred, average="macro"),
            "log_loss": ll(y, probs),
            "errores": int((pred != y).sum())}


filas = [evaluar(m, "individual", P_test[m]) for m in disponibles]
pd.DataFrame(filas).sort_values("log_loss").round(5)

## 2. Métodos de combinación

- **Votación dura:** cada modelo vota por una clase y gana la mayoría (empates: la clase con
  mayor probabilidad promedio).
- **Votación suave:** se promedian las probabilidades y gana la clase con mayor promedio.
- **Votación suave ponderada:** cada modelo pesa según su desempeño **en validación**
  (inverso de su log-loss en val).
- **Stacking:** una regresión logística aprende a combinar las probabilidades de los modelos.
  Se entrena con validación y se evalúa en test.

In [ ]:
def voto_duro(modelos, P):
    votos = np.stack([P[m].argmax(1) for m in modelos], axis=1)
    conteo = np.stack([(votos == c).sum(1) for c in range(len(CLASSES))], axis=1).astype(float)
    desempate = np.mean([P[m] for m in modelos], axis=0) * 1e-3
    return (conteo + desempate) / (conteo + desempate).sum(1, keepdims=True)


def voto_suave(modelos, P, pesos=None):
    pesos = np.ones(len(modelos)) if pesos is None else np.asarray(pesos, dtype=float)
    pesos = pesos / pesos.sum()
    return sum(w * P[m] for w, m in zip(pesos, modelos))


def pesos_val(modelos):
    return [1 / ll(y_val, P_val[m]) for m in modelos]


def stacking(X_val, X_test):
    meta = LogisticRegression(C=1.0, max_iter=2000)
    meta.fit(X_val, y_val)
    return meta.predict_proba(X_test), meta

## 3. Ensamble homogéneo: las 3 CNN

Mismo tipo de clasificador (redes convolucionales) con arquitecturas distintas.

In [ ]:
filas += [
    evaluar("CNN · voto duro", "homogéneo", voto_duro(CNN, P_test)),
    evaluar("CNN · voto suave", "homogéneo", voto_suave(CNN, P_test)),
    evaluar("CNN · voto suave ponderado", "homogéneo", voto_suave(CNN, P_test, pesos_val(CNN))),
]
print("Pesos (val):", dict(zip(CNN, np.round(np.array(pesos_val(CNN)) / sum(pesos_val(CNN)), 3))))
pd.DataFrame(filas[-3:]).round(5)

## 4. Ensamble heterogéneo: CNN + transformers

Tipos de clasificador distintos. La idea es que se equivoquen en imágenes diferentes y se
corrijan entre sí.

In [ ]:
TODOS = CNN + TRANSFORMERS
st_probs, meta_het = stacking(np.hstack([P_val[m] for m in TODOS]), np.hstack([P_test[m] for m in TODOS]))
filas += [
    evaluar("5 modelos · voto duro", "heterogéneo", voto_duro(TODOS, P_test)),
    evaluar("5 modelos · voto suave", "heterogéneo", voto_suave(TODOS, P_test)),
    evaluar("5 modelos · voto suave ponderado", "heterogéneo", voto_suave(TODOS, P_test, pesos_val(TODOS))),
    evaluar("5 modelos · stacking", "heterogéneo", st_probs),
]
print("Pesos (val):", dict(zip(TODOS, np.round(np.array(pesos_val(TODOS)) / sum(pesos_val(TODOS)), 3))))
pd.DataFrame(filas[-4:]).round(5)

## 5. Ensamble híbrido: homogéneos dentro de un heterogéneo

- **Nivel 1 (homogéneo):** voto suave de las 3 CNN, y voto suave de los 2 transformers.
- **Nivel 2 (heterogéneo):** una regresión logística recibe las salidas de los dos grupos y
  aprende cuánto confiar en cada uno según la clase.

Para estimar el desempeño del meta-clasificador sin usar test, primero se valida con 5 folds
dentro del conjunto de validación.

In [ ]:
def grupos(P):
    return np.hstack([voto_suave(CNN, P), voto_suave(TRANSFORMERS, P)])


X_val_h, X_test_h = grupos(P_val), grupos(P_test)
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
cv_probs = cross_val_predict(LogisticRegression(C=1.0, max_iter=2000), X_val_h, y_val, cv=cv,
                             method="predict_proba")
print(f"Accuracy del meta-clasificador en validación (5 folds): {accuracy_score(y_val, cv_probs.argmax(1)):.5f}")

hib_probs, meta_hib = stacking(X_val_h, X_test_h)
filas.append(evaluar("Híbrido · CNN | transformers → stacking", "híbrido", hib_probs))
pd.DataFrame(filas[-1:]).round(5)

## 6. Comparación final

In [ ]:
res = pd.DataFrame(filas)
res.to_csv(OUT_DIR / "ensambles_resultados.csv", index=False)
res.sort_values(["errores", "log_loss"]).reset_index(drop=True).round(5)

In [ ]:
orden = res.sort_values("errores", ascending=False)
colores = {"individual": "#8c8c8c", "homogéneo": "#1f77b4", "heterogéneo": "#ff7f0e", "híbrido": "#2ca02c"}
fig, ax = plt.subplots(figsize=(10, 0.42 * len(orden) + 1))
ax.barh(orden["ensamble"], orden["errores"], color=[colores[t] for t in orden["tipo"]])
for i, (e, a) in enumerate(zip(orden["errores"], orden["accuracy"])):
    ax.text(e, i, f"  {e} ({a:.2%})", va="center", fontsize=9)
ax.set_xlabel(f"Imágenes mal clasificadas en test (de {len(y_test)})")
ax.set_title("Errores en test: modelos individuales frente a ensambles")
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in colores.values()]
ax.legend(handles, colores.keys(), loc="lower right")
plt.tight_layout()
plt.show()

### ¿La mejora es real o es azar? Prueba de McNemar

Con exactitudes tan cercanas a 100 %, una diferencia de pocas imágenes puede ser casualidad.
La prueba de McNemar compara dos clasificadores **sobre las mismas imágenes de test**. Solo
cuentan los casos en que uno acierta y el otro falla. Si el valor p es menor que 0,05, la
diferencia es estadísticamente significativa.

In [ ]:
def mcnemar(probs_a, probs_b):
    a_ok, b_ok = probs_a.argmax(1) == y_test, probs_b.argmax(1) == y_test
    solo_a, solo_b = int((a_ok & ~b_ok).sum()), int((~a_ok & b_ok).sum())
    p = binomtest(min(solo_a, solo_b), solo_a + solo_b, 0.5).pvalue if solo_a + solo_b else 1.0
    return solo_a, solo_b, p


mejor_ind = res[res["tipo"] == "individual"].sort_values(["errores", "log_loss"]).iloc[0]["ensamble"]
filas_mc = []
for nombre in res.loc[res["tipo"] != "individual", "ensamble"]:
    sa, sb, p = mcnemar(PROBS[nombre], P_test[mejor_ind])
    filas_mc.append({"ensamble": nombre, "frente a": mejor_ind, "solo acierta el ensamble": sa,
                     "solo acierta el individual": sb, "valor p": p, "significativo (p<0,05)": p < 0.05})
mcn = pd.DataFrame(filas_mc)
mcn.to_csv(OUT_DIR / "ensambles_mcnemar.csv", index=False)
mcn.round(4)

## 7. Matriz de confusión e imágenes mal clasificadas del mejor ensamble

In [ ]:
mejor = res[res["tipo"] != "individual"].sort_values(["errores", "log_loss"]).iloc[0]["ensamble"]
probs_mejor = PROBS[mejor]
pred = probs_mejor.argmax(1)
print("Mejor ensamble:", mejor)

fig, ax = plt.subplots(figsize=(5.5, 5.5))
ConfusionMatrixDisplay.from_predictions(y_test, pred, labels=list(range(len(CLASSES))),
                                        display_labels=CLASSES, ax=ax, colorbar=False, xticks_rotation=45)
ax.set_title(mejor)
plt.tight_layout()
plt.show()

In [ ]:
DATA_ROOT = None
for root, dirs, _ in os.walk("/kaggle/input"):
    if set(CLASSES) <= set(dirs):
        DATA_ROOT = Path(root)
        break

errores = np.where(pred != y_test)[0]
print(f"{len(errores)} imágenes mal clasificadas de {len(y_test)}")
if DATA_ROOT is not None and len(errores):
    # test_y.npy sigue el orden de test.csv (el notebook 04 no lo reordena).
    muestra = errores[:12]
    cols = 6
    filas_img = int(np.ceil(len(muestra) / cols))
    fig, axes = plt.subplots(filas_img, cols, figsize=(2.8 * cols, 3.4 * filas_img), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for ax, i in zip(axes.ravel(), muestra):
        ax.imshow(Image.open(DATA_ROOT / test_df.loc[i, "filepath"]))
        ax.set_title(f"real: {CLASSES[y_test[i]]}\npred: {CLASSES[pred[i]]} ({probs_mejor[i].max():.0%})", fontsize=9)
    plt.suptitle("Errores del mejor ensamble")
    plt.tight_layout()
    plt.show()